# Paper 1 — Dataset downloader (Google Drive) · v3 (medium profile)

Downloads the Paper 1 datasets **directly from Hugging Face / Kaggle into**
`My Drive/Paper 1/datasets/`, already organised into folders.

**How to use**
1. `Runtime → Change runtime type → CPU` is enough (no GPU needed).
2. Run the cells top to bottom. You will be asked to approve Drive access and to paste your tokens (they are **not** saved anywhere).
3. Pick a `PROFILE` in the config cell that fits your Drive storage:

| Profile | Approx. Drive space | What you get |
|---|---|---|
| `starter` | ~6 GB (fits free 15 GB) | DiffSeg30k 3k imgs/split, SID-Set 1k/label (train) + 300/label (val), MMTD-Set annotations |
| `medium` | ~60–100 GB (needs 100 GB+ plan) | + full DiffSeg30k, SID-Set 10k/label, GenImage validation + SD v1.4 (as .tar), BR-Gen |
| `full` | several hundred GB (2 TB plan) | Everything: all 11 GenImage subsets (as .tar), full SID-Set as parquet + a 10k/label image-folder subset, full DiffSeg30k, BR-Gen, MMTD-Set |

**This copy is set to `medium`** (~60–100 GB): GenImage `Validation` + `SD_v1.4` are saved as normal image folders you can browse in Drive. Copying loose images to Drive is slow (can take many hours) but resumes after a disconnect. If you prefer speed, set `CFG['genimage_archive'] = True` to save each subset as one `.tar` instead. For training, copy the folders you need to `/content/data` first (last cell).

**Limits to plan around (full profile):** Google Drive accepts at most ~750 GB of uploads per account per day, and a free Colab session lasts up to ~12 h and stops when idle. Run the dataset cells one at a time (GenImage may need 2–3 sessions); every step resumes where it stopped.

Every step is **resumable** (already-saved files are skipped) and **stops automatically** when Drive free space drops below `MIN_FREE_GB`.
Images are written as their **original bytes** (no re-compression), which matters for forensic traces.

In [ ]:
# 1) Install libraries + mount Google Drive
import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "datasets", "huggingface_hub", "kagglehub", "pillow"], check=True)

from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/Paper 1/datasets'
os.makedirs(ROOT, exist_ok=True)
print("Datasets folder:", ROOT)

In [ ]:
# 2) Tokens (typed in hidden, kept only in this session's memory)
from getpass import getpass
os.environ['HF_TOKEN'] = getpass('Hugging Face token (hf_...): ').strip()
kg = getpass('Kaggle API token (KGAT_...), or press Enter to skip: ').strip()
if kg:
    os.environ['KAGGLE_API_TOKEN'] = kg

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print("Hugging Face login OK")

In [ ]:
# 3) CONFIG — choose a profile
PROFILE = "medium"       # "starter" | "medium" | "full"
MIN_FREE_GB = 2.0        # stop when Drive free space falls below this

GENIMAGE_SLUGS = {       # Kaggle mirror of GenImage (one dataset per generator)
    "Validation":   "vtphatt2/GenImage-Validation",
    "SD_v1.4":      "vtphatt2/GenImage-stable-diffusion-v1-4",
    "SD_v1.5":      "vtphatt2/GenImage-stable-diffusion-v1-5",
    "ADM":          "vtphatt2/GenImage-ADM",
    "BigGAN":       "vtphatt2/GenImage-BigGAN",
    "GLIDE":        "vtphatt2/GenImage-glide",
    "VQDM":         "vtphatt2/GenImage-VQDM",
    "Wukong":       "vtphatt2/GenImage-wukong",
    "Midjourney_p1":"vtphatt2/GenImage-Midjourney-Part-1",
    "Midjourney_p2":"vtphatt2/GenImage-Midjourney-Part-2",
    "Midjourney_p3":"vtphatt2/GenImage-Midjourney-Part-3",
}

PROFILES = {
    "starter": dict(diffseg_per_split=3000, sid_train_per_label=1000, sid_val_per_label=300, sid_parquet=False,
                    genimage=[], genimage_archive=True, brgen=False, mmtd=True),
    "medium":  dict(diffseg_per_split=None, sid_train_per_label=10000, sid_val_per_label=2000, sid_parquet=False,
                    genimage=["Validation", "SD_v1.4"], genimage_archive=False, brgen=True, mmtd=True),
    "full":    dict(diffseg_per_split=None, sid_train_per_label=10000, sid_val_per_label=2000, sid_parquet=True,
                    genimage=list(GENIMAGE_SLUGS), genimage_archive=True, brgen=True, mmtd=True),
}
CFG = PROFILES[PROFILE]          # None = no cap (take everything)
print(PROFILE, CFG)

In [ ]:
# 4) Helpers (space guard, safe writes, manifests)
import shutil, csv, time

class OutOfSpace(Exception):
    pass

def free_gb(path=ROOT):
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return float('inf')

def guard(extra_gb=0.0):
    f = free_gb()
    if f - extra_gb < MIN_FREE_GB:
        raise OutOfSpace(f"Only {f:.1f} GB free in Drive (limit {MIN_FREE_GB} GB) - stopping.")

def write_bytes(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.part'
    with open(tmp, 'wb') as fh:
        fh.write(data)
    os.replace(tmp, path)

def sniff_ext(b):
    if b[:8] == b'\x89PNG\r\n\x1a\n': return '.png'
    if b[:3] == b'\xff\xd8\xff':      return '.jpg'
    if b[:4] == b'RIFF' and b[8:12] == b'WEBP': return '.webp'
    if b[:2] in (b'II', b'MM'):       return '.tif'
    return '.png'

def write_csv(path, header, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(header); w.writerows(rows)

def hf_repo_size_gb(repo_id):
    from huggingface_hub import HfApi
    info = HfApi().dataset_info(repo_id, files_metadata=True)
    return sum((s.size or 0) for s in info.siblings) / 1e9

def hf_snapshot(repo_id, folder):
    from huggingface_hub import snapshot_download
    size = hf_repo_size_gb(repo_id)
    print(f"  {repo_id}: {size:.2f} GB")
    guard(extra_gb=size)
    snapshot_download(repo_id, repo_type="dataset", local_dir=f"{ROOT}/{folder}")

def run_step(name, fn, enabled=True):
    if not enabled:
        print(f"[{name}] skipped for profile '{PROFILE}'"); return
    t = time.time()
    try:
        fn()
        print(f"[{name}] done in {(time.time()-t)/60:.1f} min. Drive free: {free_gb():.1f} GB")
    except OutOfSpace as e:
        print(f"[{name}] STOPPED: {e}")
    except Exception as e:
        print(f"[{name}] FAILED: {e!r}  (re-run this cell to resume)")
print("Drive free space now:", round(free_gb(), 1), "GB")

## DiffSeg30k — locally diffusion-edited images + pixel masks (core localization set)
Source: Hugging Face `Chaos2629/Diffseg30k` (Apache-2.0).
Output: `02_DiffSeg30k/<split>/images/*`, `02_DiffSeg30k/<split>/masks/*_mask.png`, `<split>_manifest.csv`.
Mask pixel values encode the editing model (0 = background, 1 = SD2-inpainting, 2 = Kolors, 3 = SD3.5-medium, 4 = FLUX, 5 = SDXL-inpainting, 6 = GLIDE, 7 = HunyuanDiT, 8 = see dataset card). An image can have several edit turns.

In [ ]:
from datasets import load_dataset, get_dataset_split_names, Image as HFImage

def dl_diffseg():
    repo, out = "Chaos2629/Diffseg30k", f"{ROOT}/02_DiffSeg30k"
    cap = CFG['diffseg_per_split']
    for split in get_dataset_split_names(repo):
        ds = load_dataset(repo, split=split, streaming=True)
        ds = ds.cast_column("image", HFImage(decode=False)).cast_column("mask", HFImage(decode=False))
        rows, skipped = [], 0
        for i, ex in enumerate(ds):
            if cap is not None and len(rows) >= cap:
                break
            if i % 200 == 0:
                guard()
            ib, mb = (ex['image'] or {}).get('bytes'), (ex['mask'] or {}).get('bytes')
            if not ib or not mb:
                skipped += 1; continue
            stem = f"{split}_{i:06d}"
            ip = f"{out}/{split}/images/{stem}{sniff_ext(ib)}"
            mp = f"{out}/{split}/masks/{stem}_mask{sniff_ext(mb)}"
            if not os.path.exists(ip): write_bytes(ip, ib)
            if not os.path.exists(mp): write_bytes(mp, mb)
            rows.append([os.path.relpath(ip, out), os.path.relpath(mp, out)])
            if len(rows) % 500 == 0:
                print(f"  {split}: {len(rows)} saved")
        write_csv(f"{out}/{split}_manifest.csv", ["image", "mask"], rows)
        print(f"  {split}: {len(rows)} pairs saved ({skipped} skipped)")

run_step("DiffSeg30k", dl_diffseg)

## SID-Set — social-media real / fully synthetic / tampered images (+ masks for tampered)
Source: Hugging Face `saberzl/SID_Set` (CC-BY-4.0; full download ≈ 140 GB, so the profile caps images per label).
Label folders follow the SIDA paper: `0 → real`, `1 → full_synthetic`, `2 → tampered` (check the counts printed below; if one class stays at 0 the mapping differs — tell Claude).
The official **test** split is distributed separately via the SIDA GitHub repo.

In [ ]:
SID_LABELS = {0: 'real', 1: 'full_synthetic', 2: 'tampered'}

def dl_sid():
    repo, out = "saberzl/SID_Set", f"{ROOT}/03_SID-Set"
    for split, cap in [('train', CFG['sid_train_per_label']), ('validation', CFG['sid_val_per_label'])]:
        ds = load_dataset(repo, split=split, streaming=True)
        ds = ds.cast_column("image", HFImage(decode=False)).cast_column("mask", HFImage(decode=False))
        ds = ds.shuffle(seed=42, buffer_size=300)        # mixes shards so all labels appear early
        counts, rows = {k: 0 for k in SID_LABELS}, []
        for i, ex in enumerate(ds):
            lab = int(ex['label'])
            counts.setdefault(lab, 0)
            if cap is not None and counts[lab] >= cap:
                if all(counts[k] >= cap for k in SID_LABELS):
                    break
                continue
            if i % 200 == 0:
                guard()
            ib = (ex['image'] or {}).get('bytes')
            if not ib:
                continue
            name = SID_LABELS.get(lab, f'label_{lab}')
            iid = str(ex.get('img_id') or i).replace('/', '_')
            ip = f"{out}/{split}/{name}/images/{iid}{sniff_ext(ib)}"
            if not os.path.exists(ip): write_bytes(ip, ib)
            mp = ''
            mb = (ex.get('mask') or {}).get('bytes') if ex.get('mask') else None
            if lab == 2 and mb:
                mp = f"{out}/{split}/{name}/masks/{iid}_mask{sniff_ext(mb)}"
                if not os.path.exists(mp): write_bytes(mp, mb)
            counts[lab] += 1
            rows.append([os.path.relpath(ip, out), os.path.relpath(mp, out) if mp else '', lab, name])
            if len(rows) % 500 == 0:
                print(f"  {split}: {counts}")
        write_csv(f"{out}/{split}_manifest.csv", ["image", "mask", "label", "label_name"], rows)
        print(f"  {split}: saved per label {counts}")

run_step("SID-Set (image folders)", dl_sid)

### SID-Set full copy (parquet shards, ~140 GB) — `full` profile only
Stored in `03_SID-Set/parquet_full/` exactly as published (train + validation). Load later with
`load_dataset("parquet", data_files=".../parquet_full/data/train-*.parquet")`.

In [ ]:
run_step("SID-Set (full parquet)", lambda: hf_snapshot("saberzl/SID_Set", "03_SID-Set/parquet_full"),
         enabled=CFG['sid_parquet'])

## GenImage — fully AI-generated vs real (cross-generator benchmark)
Source: Kaggle mirror of GenImage (`vtphatt2/GenImage-*`, one dataset per generator; original: NeurIPS 2023, github.com/weili-thu/genimage).
Each subset is downloaded to Colab's local disk, written to Drive as one `01_GenImage/<subset>.tar`, then the local copy is deleted. Run this cell again after a disconnect: finished subsets are skipped.

**Kaggle login:** the cell uses the `KAGGLE_API_TOKEN` you entered. If it fails with an authentication error, create a *legacy* API key on kaggle.com → Settings → API, then run:
`os.environ['KAGGLE_USERNAME']='your_username'; os.environ['KAGGLE_KEY']='your_key'` and re-run this cell.

In [ ]:
def du_gb(path):
    total = 0
    for dp, _, fns in os.walk(path):
        for f in fns:
            try: total += os.path.getsize(os.path.join(dp, f))
            except OSError: pass
    return total / 1e9

def tar_to_drive(local, tar_path):
    import tarfile
    os.makedirs(os.path.dirname(tar_path), exist_ok=True)
    tmp = tar_path + '.part'
    with tarfile.open(tmp, 'w') as tf:       # uncompressed: images are already compressed
        tf.add(local, arcname=os.path.splitext(os.path.basename(tar_path))[0])
    os.replace(tmp, tar_path)

def dl_genimage():
    import kagglehub
    for name in CFG['genimage']:
        slug = GENIMAGE_SLUGS[name]
        dst = f"{ROOT}/01_GenImage/{name}"
        tar_path = f"{ROOT}/01_GenImage/{name}.tar"
        if os.path.exists(f"{dst}/_DONE") or os.path.exists(tar_path):
            print(f"  {name}: already complete, skipping"); continue
        print(f"  {name}: downloading {slug} to local disk ...")
        local = kagglehub.dataset_download(slug)
        size = du_gb(local)
        print(f"  {name}: {size:.1f} GB downloaded")
        if free_gb() - size < MIN_FREE_GB:
            print(f"  {name}: needs {size:.1f} GB but Drive has {free_gb():.1f} GB free - skipped")
            shutil.rmtree(local, ignore_errors=True); continue
        if CFG.get('genimage_archive'):
            print(f"  {name}: writing {name}.tar to Drive ...")
            tar_to_drive(local, tar_path)
            shutil.rmtree(local, ignore_errors=True)
            print(f"  {name}: saved {tar_path}")
            continue
        n = 0
        for dp, _, fns in os.walk(local):
            rel = os.path.relpath(dp, local)
            for f in fns:
                src, tgt = os.path.join(dp, f), os.path.join(dst, rel, f)
                if os.path.exists(tgt): continue
                if n % 500 == 0: guard()
                os.makedirs(os.path.dirname(tgt), exist_ok=True)
                shutil.copy2(src, tgt); n += 1
        open(f"{dst}/_DONE", 'w').write(slug)
        shutil.rmtree(local, ignore_errors=True)
        print(f"  {name}: {n} files copied to Drive")

run_step("GenImage", dl_genimage, enabled=bool(CFG['genimage']))

## MMTD-Set (FakeShield) — text explanations for tampered images
Source: Hugging Face `zhipeixu/MMTD-Set-34k`. Contains image–description conversations (JSON); the image paths refer to the original source datasets (e.g., CASIA2), which are **not** included.

In [ ]:
run_step("MMTD-Set", lambda: hf_snapshot("zhipeixu/MMTD-Set-34k", "06_MMTD-Set"), enabled=CFG['mmtd'])

## BR-Gen — scene-level local AI edits (cross-dataset localization test)
Source: Hugging Face `clpbc/BR-Gen` (AAAI 2026). For copyright reasons it ships **region masks and generated content only**; the original real photos must be fetched from their source datasets as described on the dataset card / github.com/clpbc/BR-Gen.

In [ ]:
run_step("BR-Gen", lambda: hf_snapshot("clpbc/BR-Gen", "04_BR-Gen"), enabled=CFG['brgen'])

## Manual datasets (cannot be auto-downloaded)
- **05_Chameleon** (hard real-world test set, ICLR 2025 "A Sanity Check for AI-generated Image Detection" / AIDE): download links are in the AIDE GitHub repository linked from the paper (arxiv.org/abs/2406.19435). Put the extracted folders into `05_Chameleon/`.
- **07_FaceForensics++_optional**: requires the access form at github.com/ondyari/FaceForensics. Only needed if you add a face-deepfake subset.

In [ ]:
# Inventory: file counts and sizes per dataset -> datasets/INVENTORY.csv
rows = []
for d in sorted(os.listdir(ROOT)):
    p = os.path.join(ROOT, d)
    if not os.path.isdir(p): continue
    n = sum(len(f) for _, _, f in os.walk(p))
    rows.append([d, n, round(du_gb(p), 2)])
write_csv(f"{ROOT}/INVENTORY.csv", ["dataset", "files", "size_GB"], rows)
for r in rows: print(f"{r[0]:<32}{r[1]:>9} files {r[2]:>9} GB")
print("Drive free:", round(free_gb(), 1), "GB")

## At training time: extract archives to Colab's local disk (fast)
Reading millions of small files from Drive is slow, so copy/extract what you need to `/content/data` first.

In [ ]:
# Before training: copy (folder) or extract (.tar) a GenImage subset to Colab's fast local disk
import tarfile
SUBSET = "SD_v1.4"
os.makedirs("/content/data/GenImage", exist_ok=True)
tar_src, dir_src = f"{ROOT}/01_GenImage/{SUBSET}.tar", f"{ROOT}/01_GenImage/{SUBSET}"
if os.path.exists(tar_src):
    with tarfile.open(tar_src) as tf:
        tf.extractall("/content/data/GenImage")
else:
    shutil.copytree(dir_src, f"/content/data/GenImage/{SUBSET}", dirs_exist_ok=True)
print(os.listdir(f"/content/data/GenImage/{SUBSET}"))

# Check which generators the Validation subset contains (should be all 8)
val_dir = f"{ROOT}/01_GenImage/Validation"
if os.path.isdir(val_dir):
    for dp, dns, _ in os.walk(val_dir):
        print(os.path.relpath(dp, val_dir), dns[:12]); break